# Introduction to Confidence Intervals: the Lost Key

You dropped your key somewhere along a road. You park at your best guess and agree to search a fixed distance in both directions — a bigger search distance means more confidence you'll find it, at the cost of searching more road. That is a **confidence interval**: a best guess (the sample mean) plus a buffer on each side (the **margin of error**), sized so the recipe captures the fixed-but-unknown truth (the population mean $\mu$) with a chosen **confidence level** like 95%. The subtlety the story protects you from: *the key never moves* — the randomness is all in where you parked. It is based on the opening video of Lesson 7 (Confidence Intervals).

## 1. Introduction

Lesson 5 built good samples (random, large, i.i.d.) and Lesson 6 turned them into single best numbers — point estimates. But a different sample gives a different sample mean every time, and you can never measure all of Statistopia to learn the true mean height $\mu$. The natural next question: can you use a sample mean *with a quantified degree of certainty*? Statisticians answer with an interval — a lower and an upper limit — built to contain $\mu$ with a stated probability.

After completing this notebook you will be able to:

- Tell the lost-key story and map every piece of it onto the statistical objects (key, car, search distance, road).
- Define margin of error, confidence level, and significance level $\alpha$, and relate them ($\text{confidence} = 1 - \alpha$, $\alpha/2$ per tail).
- Write the confidence-interval formula $\bar{x} \pm \text{margin of error}$ and explain the bet it expresses.
- State the correct interpretation — 95% of intervals generated by this recipe contain $\mu$ — and debunk the common wrong one.
- Simulate hundreds of confidence intervals and watch the promised coverage emerge.

## 2. Theory

### The lost key

Walking to a friend's house you drop your key somewhere along the road. The search plan: park the car at your **best guess** of where it fell, then both of you walk the same **search distance** in either direction, agreed *before* the search starts.

- Parking somewhere else shifts the whole searched stretch — but where you park is always just a guess.
- What most decides whether you find the key is the **search distance**.
- How large should it be? Decide how confident you need to be, as a percentage. Some search distance makes you 80% confident. Left the oven on and *really* need the key? Agree to a bigger distance — say one that makes you 95% confident — and accept searching much more road. Short on time? Shrink the distance, and your confidence drops with it.

**The trade-off:** higher confidence ⟺ larger search distance. The searched stretch of road is one **interval**: your best guess in the middle, a buffer on each side because you know the guess is probably wrong.

### The key never moves

The key's location is **unknown but fixed**. The *interval* is what's random — its center is a guess. So the sentence "50% of keys fall inside this interval" is nonsense: there is one key and it has always been in one place. The correct sentence is

> 50% of the **guesses** are within a search distance of the key.

The guess (where you parked) is the source of all randomness and uncertainty. Keep this picture; it is easy to lose once the road becomes a probability distribution.

### Back to Statistopia

Heights follow a normal distribution with mean $\mu$ (unknown — the key) and variance $\sigma^2$ (assumed **known** for now; knowing $\sigma$ but not $\mu$ is admittedly artificial, and later in the course both will be unknown). Take the simplest possible sample: **one person**. Their height is the sample mean $\bar{x}$, and the random variable $\bar{X}$ describing all possible sample means is identical to $X$ itself:

$$
\bar{X} \sim N(\mu, \sigma^2)
$$

Knowing this distribution doesn't reveal $\mu$ — but it lets you ask: *how far are most sample means from $\mu$?* If your sample is typical, it lies within that range too.

### Three linked numbers

- **Margin of error**: a distance on either side of $\mu$.
- **Confidence level**: the probability that a sample mean lands within the margin of error.
- **Significance level $\alpha$**: the probability that it lands *outside* — the starting point of the construction.

Choose $\alpha$ first ($\alpha = 0.05$ is the most common choice), and the confidence level is $1 - \alpha = 0.95$, i.e. 95%. Grow the margin of error until 95% of all sample means fall in the shaded middle of the curve; by the symmetry of the normal distribution, the missing $\alpha$ splits evenly — $\alpha/2 = 2.5\%$ of sample means are too big, $2.5\%$ too small.

### The confidence interval

$$
\text{confidence interval} \;=\; \bar{x} \,\pm\, \text{margin of error}
$$

The logic: *if* $\bar{x}$ is one of the 95% of sample means relatively close to $\mu$, *then* $\mu$ is also relatively close to $\bar{x}$. A confidence interval is a **bet**: "unless my sample was uncommonly large or uncommonly small, $\mu$ is within a margin of error of my sample mean."

### What 95% actually means

Generate many samples, each with its own interval $\bar{x}_i \pm \text{margin}$. Some intervals contain $\mu$ (color them green), some miss (red). At confidence level 95%, **95% of the generated intervals contain $\mu$** and 5% do not. The percentage describes the *recipe*, not any single interval: in practice you generate one interval and never learn whether it caught $\mu$ — you only know that 95% of intervals cooked up this way do.

## 3. Worked Examples

### Example 1: the dictionary

| Lost-key story | Statistics |
|---|---|
| the key (fixed, unknown location) | population mean $\mu$ |
| where you park the car | sample mean $\bar{x}$ |
| search distance | margin of error |
| searched stretch of road | confidence interval |
| "how confident do I need to be?" | confidence level $1 - \alpha$ |
| chance the key is outside the stretch | significance level $\alpha$ |

### Example 2: from $\alpha$ to the tails

With $\alpha = 0.05$: confidence level $= 1 - 0.05 = 95\%$. The normal curve is symmetric, so the 5% of "unlucky" sample means split into $2.5\%$ too large and $2.5\%$ too small — each tail holds $\alpha/2$. With $\alpha = 0.20$ instead: an 80% confidence level with $10\%$ in each tail — a shorter search distance, less road to walk, more risk of missing the key.

### Example 3: flipping the statement

The margin of error $m$ is built so that $P(\bar{X} \text{ within } m \text{ of } \mu) = 0.95$. But "$\bar{X}$ within $m$ of $\mu$" and "$\mu$ within $m$ of $\bar{X}$" are the *same event* — distance is symmetric. So before sampling, $P(\mu \in \bar{X} \pm m) = 0.95$: the random interval will capture $\mu$ 95% of the time. After sampling, the interval $\bar{x} \pm m$ is a fixed pair of numbers and $\mu$ is a fixed number — nothing is random anymore, and either it's in or it isn't. That's why the 95% belongs to the recipe, not to the one interval you happened to get.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project). The one number we need beyond the story is the margin of error for a normal distribution — for a 95% confidence level it is $1.96\,\sigma$, which we compute from the inverse error function rather than hardcode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(5)

MU_TRUE = 170.0        # Statistopia's population mean height (cm) -- "the key"
SIGMA = 10.0           # population standard deviation, assumed known

def z_for_confidence(confidence):
    """z such that a standard normal lands in [-z, z] with the given probability."""
    return float(sp.sqrt(2) * sp.erfinv(sp.Float(confidence)))

for level in (0.80, 0.95, 0.99):
    print(f"{level:.0%} confidence -> margin of error = "
          f"{z_for_confidence(level):.3f} sigma")

### One sample of one person, one interval

The video's minimal setup: measure a single random person, so $\bar{x}$ is just their height and $\bar{X} \sim N(\mu, \sigma^2)$. Build the 95% interval around it.

In [ ]:
alpha = 0.05
confidence = 1 - alpha
margin = z_for_confidence(confidence) * SIGMA

x_bar = rng.normal(MU_TRUE, SIGMA)          # the height of one random person
lower, upper = x_bar - margin, x_bar + margin

print(f"sample mean (one person's height): {x_bar:.1f} cm")
print(f"margin of error at {confidence:.0%}: {margin:.1f} cm")
print(f"confidence interval: [{lower:.1f}, {upper:.1f}]")
print(f"does it contain mu = {MU_TRUE:.0f}? {lower <= MU_TRUE <= upper}")
print("(in real life you could never check that last line -- mu is unknown)")

### The recipe keeps its promise

Generate 10,000 one-person samples, build the interval each time, and count how often it captures $\mu$. The promised 95% emerges — and so does the $\alpha/2$ split between missing high and missing low.

In [ ]:
n_trials = 10_000
sample_means = rng.normal(MU_TRUE, SIGMA, size=n_trials)

contains = np.abs(sample_means - MU_TRUE) <= margin
missed_low = sample_means < MU_TRUE - margin     # guess too small: interval below mu
missed_high = sample_means > MU_TRUE + margin

print(f"intervals containing mu: {contains.mean():.1%}   (promised: {confidence:.0%})")
print(f"missed because x-bar too small: {missed_low.mean():.2%}   (promised: {alpha/2:.1%})")
print(f"missed because x-bar too large: {missed_high.mean():.2%}   (promised: {alpha/2:.1%})")

## 5. Visualizations

### The road, the key, and three search plans

The same best guess with three different search distances: more confidence costs more road. Note the key sits where it sits — only the intervals differ.

In [ ]:
key_location = MU_TRUE                       # the key: fixed, unknown to the searcher
parked_at = 163.0                            # the best guess -- a bit off, as guesses are

fig, ax = plt.subplots(figsize=(10, 3.8))
for row, level in enumerate((0.50, 0.80, 0.95)):
    reach = z_for_confidence(level) * SIGMA
    found = abs(parked_at - key_location) <= reach
    color = "tab:green" if found else "tab:red"
    ax.plot([parked_at - reach, parked_at + reach], [row, row],
            color=color, linewidth=6, solid_capstyle="round", alpha=0.8)
    ax.annotate(f"{level:.0%} confident  (search distance {reach:.0f} m)",
                xy=(parked_at, row + 0.18), ha="center", fontsize=9)
ax.plot([parked_at], [-0.55], marker="v", color="tab:blue", markersize=10)
ax.annotate("car (best guess)", xy=(parked_at, -0.95), ha="center",
            color="tab:blue", fontsize=9)
ax.axvline(key_location, color="black", linestyle=":", linewidth=1.5)
ax.annotate("key (never moves)", xy=(key_location + 0.5, 2.6),
            fontsize=9, color="black")
ax.set_xlim(parked_at - 32, parked_at + 34)
ax.set_ylim(-1.3, 3.1)
ax.set_yticks([])
ax.set_xlabel("position along the road (m)")
ax.set_title("Bigger search distance, higher confidence -- and more road to walk")
plt.tight_layout()
plt.show()

### The margin of error on the distribution

The bell curve of $\bar{X}$, its middle 95% shaded, and the two $\alpha/2$ tails where the unlucky sample means live.

In [ ]:
heights = np.linspace(MU_TRUE - 4 * SIGMA, MU_TRUE + 4 * SIGMA, 500)
density = np.exp(-0.5 * ((heights - MU_TRUE) / SIGMA) ** 2) / (SIGMA * np.sqrt(2 * np.pi))

inside = np.abs(heights - MU_TRUE) <= margin

plt.figure(figsize=(9.5, 4.6))
plt.plot(heights, density, color="black", linewidth=1.5)
plt.fill_between(heights, density, where=inside, color="tab:green", alpha=0.4,
                 label=f"{confidence:.0%} of sample means (within the margin)")
plt.fill_between(heights, density, where=~inside, color="tab:red", alpha=0.4,
                 label=f"$\\alpha/2$ = {alpha/2:.1%} in each tail")
for x, text in [(MU_TRUE, r"$\mu$"),
                (MU_TRUE - margin, r"$\mu - $margin"),
                (MU_TRUE + margin, r"$\mu + $margin")]:
    plt.axvline(x, color="gray", linestyle=":", linewidth=1)
    plt.annotate(text, xy=(x, plt.ylim()[1] * 0.02), ha="center", fontsize=10)
plt.xlabel(r"sample mean $\bar{x}$ (cm)")
plt.ylabel("density")
plt.title(r"Grow the margin of error until 95% of sample means fall inside")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The video's centerpiece: 100 intervals, green and red

One hundred one-person samples, each with its 95% interval. Green intervals caught $\mu$; red ones missed. About 95 should be green — and remember, in practice you get exactly one of these and no color.

In [ ]:
n_intervals = 100
means_100 = rng.normal(MU_TRUE, SIGMA, size=n_intervals)
caught = np.abs(means_100 - MU_TRUE) <= margin

fig, ax = plt.subplots(figsize=(10, 6.5))
for i, (m, ok) in enumerate(zip(means_100, caught)):
    color = "tab:green" if ok else "tab:red"
    ax.plot([m - margin, m + margin], [i, i], color=color,
            linewidth=1.6, alpha=0.85)
    ax.plot([m], [i], marker="o", color=color, markersize=2.5)
ax.axvline(MU_TRUE, color="black", linestyle="--", linewidth=1.5,
           label=rf"$\mu = {MU_TRUE:.0f}$ (fixed, unknown)")
ax.set_xlabel("height (cm)")
ax.set_ylabel("interval number")
ax.set_title(f"{caught.sum()} of {n_intervals} intervals contain "
             r"$\mu$ -- the 95% describes the recipe, not one interval")
ax.grid(True, alpha=0.3, axis="x")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### The whole story on one page

Everything above in a single sheet: where the sample mean lands, 100 intervals and the few that miss, the lost-key dictionary, and what the 95% really promises.

<img src="img/01_intro_confidence_intervals.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** For significance levels $\alpha = 0.10$ and $\alpha = 0.01$: what is the confidence level of each, and how much probability sits in each tail?

**Exercise 2.** A friend computes the interval $[165.2, 174.8]$ for Statistopia's mean height and says: "95% of population means fall inside my interval." Rewrite the sentence correctly, using the lost-key story to explain what is fixed and what is random.

### Intermediate

**Exercise 3.** Simulate 10,000 one-person samples and verify empirically that the 80% and the 99% recipes cover $\mu$ at (about) their stated rates. Report each coverage and each tail-miss rate.

**Exercise 4.** *The oven is on.* You need to be 99.9% confident instead of 95%. Compute the margin of error (in multiples of $\sigma$ and in cm) and compare the interval widths. What did the extra certainty cost?

### Advanced

**Exercise 5.** *More people, less road.* Lesson 5 showed that a sample of $n$ people has $\bar{X} \sim N(\mu, \sigma^2/n)$, so the margin of error shrinks by $\sqrt{n}$. Verify by simulation: for $n = 1, 4, 25, 100$, build 95% intervals from $n$-person samples and confirm the coverage stays 95% while the width drops as $1/\sqrt{n}$.

**Exercise 6.** *The price of certainty.* Plot the required margin of error (in multiples of $\sigma$) as a function of the confidence level from 50% to 99.99%. Where does the curve explode, and what does that say about demanding (near-)perfect confidence?

## 7. Solutions

In [ ]:
# Exercise 1: alpha -> confidence level and tails
for a in (0.10, 0.01):
    print(f"alpha = {a:>4}: confidence level = {1 - a:.0%}, "
          f"each tail holds alpha/2 = {a / 2:.1%}")

In [ ]:
# Exercise 2: the corrected sentence
print("Exercise 2: 'The population mean is one fixed number -- like the key, it")
print("never moves, and it is either inside [165.2, 174.8] or it is not. What is")
print("random is my interval, because its center is a guess (my sample mean).")
print("95% of intervals built by this recipe contain the population mean.'")

In [ ]:
# Exercise 3: coverage at 80% and 99%
for level in (0.80, 0.99):
    m = z_for_confidence(level) * SIGMA
    xbars = rng.normal(MU_TRUE, SIGMA, size=10_000)
    cover = (np.abs(xbars - MU_TRUE) <= m).mean()
    low = (xbars < MU_TRUE - m).mean()
    high = (xbars > MU_TRUE + m).mean()
    print(f"{level:.0%} recipe: coverage {cover:.1%}  "
          f"(missed low {low:.2%}, missed high {high:.2%}, "
          f"promised tails {(1 - level) / 2:.2%} each)")

In [ ]:
# Exercise 4: 99.9% confidence
for level in (0.95, 0.999):
    z = z_for_confidence(level)
    print(f"{level:.1%} confidence: margin = {z:.2f} sigma = {z * SIGMA:.1f} cm, "
          f"interval width = {2 * z * SIGMA:.1f} cm")
ratio = z_for_confidence(0.999) / z_for_confidence(0.95)
print(f"\nThe extra certainty widens the interval by {(ratio - 1):.0%}: "
      "a much longer stretch of road to search.")

In [ ]:
# Exercise 5: bigger samples, narrower intervals, same coverage
z95 = z_for_confidence(0.95)
print(f"{'n':>4s} {'margin (cm)':>12s} {'coverage':>9s} {'width vs n=1':>13s}")
margin_n1 = z95 * SIGMA
for n in (1, 4, 25, 100):
    m_n = z95 * SIGMA / np.sqrt(n)
    xbars = rng.normal(MU_TRUE, SIGMA / np.sqrt(n), size=10_000)
    cover = (np.abs(xbars - MU_TRUE) <= m_n).mean()
    print(f"{n:>4d} {m_n:>12.1f} {cover:>9.1%} {m_n / margin_n1:>13.2f}")
print("\nCoverage holds at ~95% while the width shrinks like 1/sqrt(n):")
print("more people buys a shorter search distance at the same confidence.")

In [ ]:
# Exercise 6: the price of certainty
levels = np.linspace(0.50, 0.9999, 300)
z_values = np.array([z_for_confidence(c) for c in levels])

plt.figure(figsize=(9, 4.4))
plt.plot(levels * 100, z_values, color="tab:purple", linewidth=2)
for level in (0.80, 0.95, 0.99):
    z = z_for_confidence(level)
    plt.plot(level * 100, z, "ko", markersize=4)
    plt.annotate(f"{level:.0%}: {z:.2f}$\\sigma$",
                 xy=(level * 100, z), xytext=(-10, 10),
                 textcoords="offset points", fontsize=9, ha="right")
plt.xlabel("confidence level (%)")
plt.ylabel(r"required margin of error (multiples of $\sigma$)")
plt.title("Exercise 6: certainty gets expensive -- the margin explodes near 100%")
plt.grid(True, alpha=0.3)
plt.show()

print("Absolute certainty would require an infinite search distance: the whole")
print("road. Useful intervals always accept some risk of missing the key.")

## 8. Conclusion

- A **confidence interval** is a best guess with a buffer: $\bar{x} \pm \text{margin of error}$ — the parked car plus an agreed search distance in each direction.
- Three linked numbers: choose the **significance level** $\alpha$ (commonly 0.05), get the **confidence level** $1 - \alpha$, and grow the **margin of error** until that share of sample means falls inside — symmetry puts $\alpha/2$ in each tail.
- **The key never moves**: $\mu$ is fixed and unknown; the interval is random because its center is a guess. Say "95% of the guesses are within a search distance of the key," never "95% of keys fall in my interval."
- The 95% describes the **recipe**: of many intervals generated this way, ~95% contain $\mu$ — but in practice you generate one, and can never check it.
- The trade-offs, quantified in the exercises: more confidence costs a wider interval (explosively so near 100%), and more data buys narrower intervals ($\propto 1/\sqrt{n}$) at the same confidence.

Next: computing the margin of error exactly — and what to do once the sample has more than one person in it.